In [2]:
import time
from statistics import mean
from robomaster import robot

# ===================== CONFIG =====================
CONN_TYPE = "ap"          # "ap" ต่อ AP หุ่นโดยตรง | "sta" ผ่านเราเตอร์
FREQ_HZ = 10               # ความถี่ที่ subscribe ToF (ครั้ง/วินาที)
SAMPLE_PER_POSE = 3        # อ่านค่ากี่ครั้งต่อหนึ่ง "มุม" (ปรับเพิ่มเพื่อลด noise)
DWELL_SEC = 0.10           # หน่วงหลังหมุน gimbal เพื่อรอหยุดนิ่ง (วินาที)
GIMBAL_SPEED = 200         # ความเร็วหมุน gimbal (deg/s) เร็วขึ้นเพื่อให้ทัน 5 วินาที
SCAN_WINDOW_SEC = 5.0      # ระยะเวลาการสแกนต่อรอบ (วินาที)

# ชุดมุม yaw (องศา) ของแต่ละทิศ
ANGLES_RIGHT = [90]
ANGLES_FRONT = [0]
ANGLES_LEFT  = [-90]

# ค่ากำหนดการตัดสินใจ (เมตร)
SIDE_CLOSE_TH = 0.2       # เกณฑ์ "ชิดผนัง" ด้านซ้าย/ขวา
FRONT_NEAR_MIN = 0.20      # เกณฑ์หน้าใกล้ (ต่ำกว่านี้ให้หลบก่อน)
FRONT_NEAR_MAX = 0.50

# ระยะที่จะสั่งเคลื่อนที่ (เมตร)
STEP_FORWARD_NARROW = 0.25 # ใช้ในกรณีซ้ายและขวาแคบทั้งคู่ (เดินหน้า)
STEP_STRAFE_EDGE   = 0.10  # ใช้ในกรณีชิดข้างใดข้างหนึ่ง -> สไตรฟ์ไปด้านตรงข้าม
TURN_DEG = 90              # องศาที่จะเลี้ยวซ้าย/ขวาเมื่อหน้าติด (ปรับได้)

# ความเร็วการเคลื่อนที่ของ chassis
XY_SPEED = 1               # ความเร็วเคลื่อนที่เชิงเส้น (m/s)
Z_SPEED  = 60              # ความเร็วการหมุน (deg/s)

# เก็บค่าดิบล่าสุดของ ToF1 (mm) จาก callback
latest_tof1_mm = None

# ===================== CALLBACK =====================
def tof_cb(sub_info):
    """
    ถูกเรียกอัตโนมัติเมื่อหุ่นส่งข้อมูล ToF (หน่วย mm) มาใหม่
    sub_info: [tof1, tof2, tof3, tof4] -> เราใช้เฉพาะ tof1 (index 0)
    """
    global latest_tof1_mm
    latest_tof1_mm = float(sub_info[0])  # เก็บค่าดิบ (mm)

# ===================== SCAN HELPERS =====================
def measure_at_angle(ep_gimbal, yaw_deg):
    """
    หมุน gimbal ไปยังมุม yaw_deg (องศา) แล้วอ่านค่า ToF1 หลายครั้ง
    คืนค่า: ระยะ "เฉลี่ย" ของมุมนี้ในหน่วย "เมตร (m)"
    """
    # หมุน gimbal (pitch = 0 เสมอ) แล้วรอให้หยุดนิ่ง
    ep_gimbal.moveto(pitch=0, yaw=yaw_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(DWELL_SEC)

    # อ่านค่าดิบ mm ซ้ำ ๆ แล้วเฉลี่ย -> แปลงเป็น m
    mm_list = []
    for _ in range(SAMPLE_PER_POSE):
        if latest_tof1_mm is not None:
            mm_list.append(latest_tof1_mm)
        time.sleep(1.0 / max(1, FREQ_HZ))

    if not mm_list:
        return None
    return mean(mm_list) / 1000.0  # mm -> m

def measure_direction(ep_gimbal, angles):
    """
    วัดระยะสำหรับ "หนึ่งทิศ" โดยวนวัดทุกมุมใน angles แล้วเฉลี่ยอีกชั้น
    """
    vals_m = []
    for ang in angles:
        d_m = measure_at_angle(ep_gimbal, ang)
        if d_m is not None:
            vals_m.append(d_m)
    if not vals_m:
        return None
    return mean(vals_m)

def scan_window(ep_gimbal, window_sec=SCAN_WINDOW_SEC):
    """
    สแกนภายในกรอบเวลา window_sec (~5 วินาที):
    - ทำการ "ครบชุดทิศ" (Right -> Front -> Left) กี่ครั้งก็ได้เท่าที่เวลาพอ
    - สุดท้ายเฉลี่ยค่าของแต่ละทิศจากทุกครั้งที่วัดได้ในช่วงเวลา
    เหตุผล: ถ้าบางจังหวะอ่านไม่ทัน/แกว่ง จะมีหลายตัวอย่างช่วยให้ค่ากลางนิ่งขึ้น
    """
    t0 = time.time()
    buf_right, buf_front, buf_left = [], [], []

    while time.time() - t0 < window_sec:
        d_right = measure_direction(ep_gimbal, ANGLES_RIGHT)
        d_front = measure_direction(ep_gimbal, ANGLES_FRONT)
        d_left  = measure_direction(ep_gimbal, ANGLES_LEFT)

        if d_right is not None: buf_right.append(d_right)
        if d_front is not None: buf_front.append(d_front)
        if d_left  is not None: buf_left.append(d_left)

        # ถ้าเวลาใกล้ครบแล้ว ก็ออกจากลูป
        if time.time() - t0 >= window_sec:
            break

    # ถ้าไม่มีข้อมูลเลย ให้ลองวัดด่วนรอบเดียว (กันกรณีพลาด)
    if not buf_right and not buf_front and not buf_left:
        d_right = measure_direction(ep_gimbal, ANGLES_RIGHT)
        d_front = measure_direction(ep_gimbal, ANGLES_FRONT)
        d_left  = measure_direction(ep_gimbal, ANGLES_LEFT)
        if d_right is not None: buf_right.append(d_right)
        if d_front is not None: buf_front.append(d_front)
        if d_left  is not None: buf_left.append(d_left)

    # เฉลี่ยค่าแต่ละทิศ (ถ้าไม่มีข้อมูลทิศใดเลยจะเป็น None)
    avg_right = mean(buf_right) if buf_right else None
    avg_front = mean(buf_front) if buf_front else None
    avg_left  = mean(buf_left)  if buf_left  else None

    # หมุน gimbal กลับหน้า เพื่อพร้อมสำหรับเคลื่อน/รอบถัดไป
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    return avg_right, avg_front, avg_left

# ===================== DECISION & MOTION =====================
def decide_and_move(ep_chassis, ep_gimbal, d_right, d_front, d_left):
    """
    ตัดสินใจและสั่งเคลื่อนที่ตามเงื่อนไขที่กำหนด
    - ทุกคำสั่ง move/turn จะ .wait_for_completed() เพื่อให้จบก่อนคาบถัดไป
    - หลัง "เลี้ยว" จะ recenter gimbal เพื่อให้หันไปด้านหน้าทิศใหม่เสมอ (หันตามการเลี้ยว)
    """
    # ความปลอดภัย: ถ้าหน้าใกล้มากกว่าเกณฑ์ (front < 0.2 m) -> ห้ามเดินหน้า ให้เลี้ยวหลบ
    if d_front is not None and d_front < FRONT_NEAR_MIN:
        # เลือกเลี้ยวไปด้านที่ "กว้างกว่า" (ระยะมากกว่า)
        turn_dir = "left" if (d_left or 0) > (d_right or 0) else "right"
        print(f"[SAFETY] Front={d_front:.2f} m ใกล้มาก! เลี้ยว{turn_dir} {TURN_DEG}°")
        if turn_dir == "left":
            ep_chassis.move(x=0, y=0, z=+TURN_DEG, z_speed=Z_SPEED).wait_for_completed()
        else:
            ep_chassis.move(x=0, y=0, z=-TURN_DEG, z_speed=Z_SPEED).wait_for_completed()
        ep_gimbal.recenter().wait_for_completed()
        return  # จบการตัดสินใจรอบนี้

    # เงื่อนไข 4 และ 5: ถ้าหน้าอยู่ในช่วง 0.2–0.5 m ให้เลี้ยวตามด้านชิด
    if d_front is not None and (FRONT_NEAR_MIN <= d_front <= FRONT_NEAR_MAX):
        if d_right is not None and d_right < SIDE_CLOSE_TH:
            print(f"[TURN LEFT] Front={d_front:.2f} m & Right={d_right:.2f} m<0.3 -> เลี้ยวซ้าย {TURN_DEG}°")
            ep_chassis.move(x=0, y=0, z=+TURN_DEG, z_speed=Z_SPEED).wait_for_completed()
            ep_gimbal.recenter().wait_for_completed()
            return
        if d_left is not None and d_left < SIDE_CLOSE_TH:
            print(f"[TURN RIGHT] Front={d_front:.2f} m & Left={d_left:.2f} m<0.3 -> เลี้ยวขวา {TURN_DEG}°")
            ep_chassis.move(x=0, y=0, z=-TURN_DEG, z_speed=Z_SPEED).wait_for_completed()
            ep_gimbal.recenter().wait_for_completed()
            return

    # เงื่อนไข 1: ซ้าย<0.3 และ ขวา<0.3 -> เดินหน้า 0.3 m
    if (d_left is not None and d_left < SIDE_CLOSE_TH) and (d_right is not None and d_right < SIDE_CLOSE_TH):
        print(f"[FWD 0.30] Left={d_left:.2f} m & Right={d_right:.2f} m < 0.3")
        ep_chassis.move(x=+STEP_FORWARD_NARROW, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed()
        return

    # เงื่อนไข 2 (ปรับ): ซ้ายชิด (<0.3) และขวากว้าง (>0.3) -> สไตรฟ์ขวา 0.10 m
    if (d_left is not None and d_left < SIDE_CLOSE_TH) and (d_right is not None and d_right > SIDE_CLOSE_TH):
        print(f"[STRAFE RIGHT 0.10] Left={d_left:.2f} m <0.3 & Right={d_right:.2f} m >0.3")
        ep_chassis.move(x=0, y=+STEP_STRAFE_EDGE, z=0, xy_speed=XY_SPEED).wait_for_completed()
        return

    # เงื่อนไข 3 (ปรับ): ขวาชิด (<0.3) และซ้ายกว้าง (>0.3) -> สไตรฟ์ซ้าย 0.10 m
    if (d_right is not None and d_right < SIDE_CLOSE_TH) and (d_left is not None and d_left > SIDE_CLOSE_TH):
        print(f"[STRAFE LEFT 0.10] Right={d_right:.2f} m <0.3 & Left={d_left:.2f} m >0.3")
        ep_chassis.move(x=0, y=-STEP_STRAFE_EDGE, z=0, xy_speed=XY_SPEED).wait_for_completed()
        return

    # กรณีไม่เข้าเงื่อนไขใดเลย -> เดินหน้าเบา ๆ 0.2 m (ค่าเริ่มต้น)
    print(f"[FWD 0.25 DEFAULT] R={d_right:.2f} F={d_front:.2f} L={d_left:.2f}")
    ep_chassis.move(x=0.25, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed()

# ===================== MAIN =====================
def main():
    # เชื่อมต่อหุ่นและโมดูลย่อย
    ep = robot.Robot()
    ep.initialize(conn_type=CONN_TYPE)
    ep_chassis = ep.chassis
    ep_gimbal  = ep.gimbal
    ep_sensor  = ep.sensor

    # ตั้ง gimbal หันหน้าไว้ก่อน
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # สมัครข้อมูล ToF
    ep_sensor.sub_distance(freq=FREQ_HZ, callback=tof_cb)

    print("เริ่มลูป: สแกน 5 วิ → ตัดสินใจ → เคลื่อนที่ → ทำซ้ำ (Ctrl+C เพื่อหยุด)")
    try:
        while True:
            # 1) สแกนภายใน 5 วินาที ได้ค่าเฉลี่ยของแต่ละทิศ (เมตร)
            d_right, d_front, d_left = scan_window(ep_gimbal, SCAN_WINDOW_SEC)
            if None in (d_right, d_front, d_left):
                print("[WARN] สแกนไม่ครบทุกทิศ ลองใหม่ในรอบถัดไป")
                continue

            print(f"[SCAN AVG] Right={d_right:.2f} m | Front={d_front:.2f} m | Left={d_left:.2f} m")

            # 2) ตัดสินใจและสั่งเคลื่อนที่ตามเงื่อนไข
            decide_and_move(ep_chassis, ep_gimbal, d_right, d_front, d_left)

            # 3) (ออปชัน) เว้นระยะสั้น ๆ ก่อนเริ่มสแกนรอบใหม่
            time.sleep(0.2)

    except KeyboardInterrupt:
        print("หยุดตามคำสั่งผู้ใช้")
    finally:
        # เลิก subscribe + recenter + ปิดการเชื่อมต่อ
        try:
            ep_sensor.unsub_distance()
        except Exception:
            pass
        try:
            ep_gimbal.recenter().wait_for_completed()
        except Exception:
            pass
        ep.close()

if __name__ == "__main__":
    main()

เริ่มลูป: สแกน 5 วิ → ตัดสินใจ → เคลื่อนที่ → ทำซ้ำ (Ctrl+C เพื่อหยุด)
[SCAN AVG] Right=0.21 m | Front=1.31 m | Left=0.24 m
[FWD 0.25 DEFAULT] R=0.21 F=1.31 L=0.24
[SCAN AVG] Right=0.15 m | Front=0.99 m | Left=0.22 m
[STRAFE LEFT 0.10] Right=0.15 m <0.3 & Left=0.22 m >0.3
[SCAN AVG] Right=0.28 m | Front=1.00 m | Left=0.14 m
[STRAFE RIGHT 0.10] Left=0.14 m <0.3 & Right=0.28 m >0.3
[SCAN AVG] Right=0.20 m | Front=1.01 m | Left=0.21 m
[FWD 0.25 DEFAULT] R=0.20 F=1.01 L=0.21
หยุดตามคำสั่งผู้ใช้


In [12]:
from robomaster import robot
import time
import os
import csv

# ===== PID Parameters =====
Kp = 2.0
Ki = 0.1
Kd = 0.05
V_MAX = 0.8
V_MIN = -0.8

TOF_OFFSET_M = 0.4  # target = tof_m - offset
TARGET_MIN = 0.10    # m
TARGET_MAX = 2.50    # m

# ===== Globals =====
current_x = 0.0
latest_tof1_mm = None

# ===== Helpers =====
def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))

# ตำแหน่งจาก odometry
def pos_info_handler(position_info):
    global current_x
    current_x = position_info[0]

# ค่า ToF ด้านหน้า
def tof_cb(sub_info):
    global latest_tof1_mm
    try:
        latest_tof1_mm = float(sub_info[0])
    except:
        latest_tof1_mm = None

def get_dynamic_target():
    """
    target = tof_m - TOF_OFFSET_M
    ลิมิตในช่วง TARGET_MIN..TARGET_MAX
    """
    fallback = 1.2
    if latest_tof1_mm is None or latest_tof1_mm <= 0:
        td = fallback
        src = "fallback"
    else:
        tof_m = latest_tof1_mm / 1000.0
        td = max(TARGET_MIN, min(TARGET_MAX, tof_m - TOF_OFFSET_M))
        src = f"tof={tof_m:.2f}m"
    print(f"[TARGET] {td:.2f} m ({src})")
    return td

# เดินหน้าด้วย PID
def move_forward_pid(ep_chassis, distance):
    global current_x
    print(f"Moving forward {distance:.2f} m using PID...")
    integral = 0.0
    previous_error = 0.0
    dt = 0.1

    start_pos = current_x

    while True:
        meas = (current_x - start_pos)
        error = distance - meas

        if abs(error) <= 0.01:
            ep_chassis.drive_speed(x=0, y=0, z=0)
            print("Reached target distance.")
            break

        integral += error * dt
        derivative = (error - previous_error) / dt
        previous_error = error

        speed_cmd = (Kp * error) + (Ki * integral) + (Kd * derivative)
        speed_cmd = clamp(speed_cmd, V_MIN, V_MAX)

        ep_chassis.drive_speed(x=speed_cmd, y=0, z=0)
        print(f"Error={error:.3f}, Speed={speed_cmd:.3f}, Meas={meas:.3f}")

        time.sleep(dt)

# ===== Main =====
if __name__ == '__main__':
    ep_robot = robot.Robot()
    ep_robot.initialize(conn_type="ap")

    ep_chassis = ep_robot.chassis
    ep_sensor  = ep_robot.sensor

    # สมัคร callback
    ep_chassis.sub_position(freq=10, callback=pos_info_handler)
    ep_sensor.sub_distance(freq=10, callback=tof_cb)

    try:
        time.sleep(1)  # รอให้มีข้อมูล ToF
        target_distance = get_dynamic_target()
        move_forward_pid(ep_chassis, distance=target_distance)

    finally:
        ep_chassis.drive_speed(x=0, y=0, z=0)
        ep_chassis.unsub_position()
        ep_sensor.unsub_distance()
        ep_robot.close()
        print("Robot closed.")


[TARGET] 0.99 m (tof=1.39m)
Moving forward 0.99 m using PID...
Error=0.988, Speed=0.800, Meas=0.000
Error=0.988, Speed=0.800, Meas=0.000
Error=0.988, Speed=0.800, Meas=0.000
Error=0.984, Speed=0.800, Meas=0.004
Error=0.942, Speed=0.800, Meas=0.046
Error=0.858, Speed=0.800, Meas=0.130
Error=0.759, Speed=0.800, Meas=0.229
Error=0.666, Speed=0.800, Meas=0.322
Error=0.583, Speed=0.800, Meas=0.405
Error=0.394, Speed=0.776, Meas=0.594
Error=0.394, Speed=0.800, Meas=0.594
Error=0.311, Speed=0.669, Meas=0.677
Error=0.311, Speed=0.714, Meas=0.677
Error=0.311, Speed=0.717, Meas=0.677
Error=0.023, Speed=-0.002, Meas=0.965
Error=-0.044, Speed=-0.028, Meas=1.032
Error=-0.117, Speed=-0.177, Meas=1.105
Error=-0.159, Speed=-0.246, Meas=1.147
Error=-0.158, Speed=-0.224, Meas=1.146
Error=-0.134, Speed=-0.167, Meas=1.122
Error=-0.134, Speed=-0.180, Meas=1.122
Error=-0.134, Speed=-0.182, Meas=1.122
Error=-0.037, Speed=0.060, Meas=1.025
Error=-0.021, Speed=0.052, Meas=1.009
Reached target distance.
Robot c

In [1]:
import time
from statistics import mean
from robomaster import robot

# ===================== CONFIG =====================
CONN_TYPE = "ap"

FREQ_HZ = 10               # ความถี่รับ ToF
SAMPLE_PER_POSE = 3        # อ่านซ้ำเพื่อลด noise
DWELL_SEC = 0.10           # เวลารอให้ gimbal นิ่งก่อนอ่าน
GIMBAL_SPEED = 200         # deg/s
STEP_FORWARD = 0.30        # m เดินหน้าหลังหัน (ตั้ง 0 ถ้าไม่อยากเดิน)
XY_SPEED = 0.6             # m/s
Z_SPEED  = 90              # deg/s

# เซ็ตมุม (องศา) สำหรับแต่ละทิศ (เฉลี่ยหลายมุมให้ค่านิ่ง)
ANGLES_LEFT  = [-90]
ANGLES_FRONT = [0]
ANGLES_RIGHT = [90]

# ===================== GLOBAL =====================
latest_tof1_mm = None  # ToF1 ด้านหน้า (mm)

# ===================== CALLBACK =====================
def tof_cb(sub_info):
    global latest_tof1_mm
    # sub_info = [tof1, tof2, tof3, tof4] หน่วย mm
    try:
        latest_tof1_mm = float(sub_info[0])
    except Exception:
        latest_tof1_mm = None

# ===================== MEASURE HELPERS =====================
def measure_at_angle(ep_gimbal, yaw_deg):
    """หมุน gimbal ไป yaw_deg แล้วอ่าน ToF1 หลายครั้ง -> คืนค่าเป็นเมตร"""
    ep_gimbal.moveto(pitch=0, yaw=yaw_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(DWELL_SEC)

    samples = []
    for _ in range(SAMPLE_PER_POSE):
        if latest_tof1_mm is not None and latest_tof1_mm > 0:
            samples.append(latest_tof1_mm)
        time.sleep(1.0 / max(1, FREQ_HZ))

    if not samples:
        return None
    return mean(samples) / 1000.0  # mm -> m

def measure_direction(ep_gimbal, angles):
    """วัดหนึ่งทิศโดยเฉลี่ยค่าจากหลายมุม"""
    vals = []
    for ang in angles:
        d = measure_at_angle(ep_gimbal, ang)
        if d is not None:
            vals.append(d)
    return mean(vals) if vals else None

def scan_three(ep_gimbal):
    """สแกนซ้าย-หน้า-ขวา แล้วคืนระยะเฉลี่ย (m) ของแต่ละทิศ"""
    d_left  = measure_direction(ep_gimbal, ANGLES_LEFT)
    d_front = measure_direction(ep_gimbal, ANGLES_FRONT)
    d_right = measure_direction(ep_gimbal, ANGLES_RIGHT)

    # หลังสแกน หมุน gimbal กลับหน้า
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    return d_left, d_front, d_right

# ===================== DECIDE & TURN =====================
def choose_and_face_best(ep_chassis, ep_gimbal):
    """
    1) สแกนซ้าย-หน้า-ขวา
    2) เลือกทิศที่ไกลสุด
    3) หมุนหุ่นไปทิศนั้น พร้อมตั้ง gimbal ให้หันหน้าไปทางเดียวกัน (recenter หลังหมุน)
    4) (ออปชัน) เดินหน้า STEP_FORWARD เมตร
    """
    d_left, d_front, d_right = scan_three(ep_gimbal)
    print(f"[SCAN] Left={d_left} m | Front={d_front} m | Right={d_right} m")

    # หา max ที่ไม่เป็น None
    candidates = []
    if d_left  is not None:  candidates.append(("left",  d_left))
    if d_front is not None:  candidates.append(("front", d_front))
    if d_right is not None:  candidates.append(("right", d_right))

    if not candidates:
        print("[WARN] ไม่มีข้อมูล ToF ใช้การ: ไม่หมุน/ไม่เดิน")
        return

    best_dir, best_dist = max(candidates, key=lambda x: x[1])
    print(f"[DECIDE] ไปทาง '{best_dir}' (ไกลสุด ~ {best_dist:.2f} m)")

    # map ทิศ -> องศาที่จะหมุน (relative to current heading)
    turn_deg = {"left": +90, "front": 0, "right": -90}[best_dir]

    # หัน gimbal ไปทิศเป้าหมายก่อน (เพื่อให้คนดูรู้ว่าจะไปทางไหน)
    ep_gimbal.moveto(pitch=0, yaw=turn_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # หมุนหุ่นไปทิศเดียวกัน
    if turn_deg != 0:
        ep_chassis.move(x=0, y=0, z=turn_deg, z_speed=Z_SPEED).wait_for_completed()

    # หลังหมุนเสร็จ ให้ gimbal recenter เพื่อหันหน้าไปกับตัวหุ่น
    ep_gimbal.recenter().wait_for_completed()

    # (ออปชัน) เดินหน้าเล็กน้อยเพื่อตามทิศที่เลือก
    if STEP_FORWARD > 0:
        ep_chassis.move(x=STEP_FORWARD, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed()

# ===================== MAIN =====================
def main():
    ep = robot.Robot()
    ep.initialize(conn_type=CONN_TYPE)

    ep_chassis = ep.chassis
    ep_gimbal  = ep.gimbal
    ep_sensor  = ep.sensor

    # ตั้งค่าเริ่มต้น
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # สมัคร ToF
    ep_sensor.sub_distance(freq=FREQ_HZ, callback=tof_cb)

    print("เริ่ม: สแกนซ้าย-หน้า-ขวา → เลือกทิศที่ไกลสุด → หมุนหุ่นและ gimbal ไปทางนั้น → (ออปชัน) เดินหน้า")
    try:
        while True:
            choose_and_face_best(ep_chassis, ep_gimbal)
            time.sleep(0.3)  # เว้นช่วงก่อนสแกนรอบต่อไป

    except KeyboardInterrupt:
        print("หยุดตามคำสั่งผู้ใช้")

    finally:
        try:
            ep_sensor.unsub_distance()
        except Exception:
            pass
        try:
            ep_gimbal.recenter().wait_for_completed()
        except Exception:
            pass
        ep.close()

if __name__ == "__main__":
    main()


เริ่ม: สแกนซ้าย-หน้า-ขวา → เลือกทิศที่ไกลสุด → หมุนหุ่นและ gimbal ไปทางนั้น → (ออปชัน) เดินหน้า
[SCAN] Left=0.15266666666666664 m | Front=1.6033333333333333 m | Right=0.202 m
[DECIDE] ไปทาง 'front' (ไกลสุด ~ 1.60 m)
[SCAN] Left=0.157 m | Front=1.279 m | Right=0.252 m
[DECIDE] ไปทาง 'front' (ไกลสุด ~ 1.28 m)
[SCAN] Left=0.19966666666666666 m | Front=0.9753333333333334 m | Right=0.26166666666666666 m
[DECIDE] ไปทาง 'front' (ไกลสุด ~ 0.98 m)
[SCAN] Left=0.199 m | Front=0.6506666666666666 m | Right=0.279 m
[DECIDE] ไปทาง 'front' (ไกลสุด ~ 0.65 m)
[SCAN] Left=0.16833333333333333 m | Front=0.073 m | Right=1.549 m
[DECIDE] ไปทาง 'right' (ไกลสุด ~ 1.55 m)
[SCAN] Left=0.09233333333333332 m | Front=1.234 m | Right=1.597 m
[DECIDE] ไปทาง 'right' (ไกลสุด ~ 1.60 m)
[SCAN] Left=1.223 m | Front=1.3933333333333333 m | Right=0.4956666666666667 m
[DECIDE] ไปทาง 'front' (ไกลสุด ~ 1.39 m)
[SCAN] Left=0.21866666666666665 m | Front=1.0843333333333331 m | Right=0.18133333333333335 m
[DECIDE] ไปทาง 'front' (ไ

In [2]:
import time
from statistics import mean
from math import hypot
from robomaster import robot

# ===================== CONFIG =====================
CONN_TYPE = "ap"

# --- Gimbal / Scan ---
FREQ_HZ = 10               # ความถี่รับ ToF
SAMPLE_PER_POSE = 3        # อ่านซ้ำเพื่อลด noise
DWELL_SEC = 0.10           # รอ gimbal นิ่งก่อนอ่าน
GIMBAL_SPEED = 200         # deg/s
Z_SPEED  = 90              # deg/s หมุนตัวหุ่น

ANGLES_LEFT  = [-90]
ANGLES_FRONT = [0]
ANGLES_RIGHT = [90]

# --- PID forward (body-x) ---
Kp = 2.0
Ki = 0.1
Kd = 0.05
V_MAX = 0.8        # m/s
V_MIN = -0.8       # m/s
DT   = 0.10        # s (คาบควบคุม)

# เป้าหมายเดินหน้า = ToF(front) - OFFSET
TOF_OFFSET_M = 0.30
TARGET_MIN   = 0.10  # m
TARGET_MAX   = 2.50  # m

# Safety ระหว่างวิ่ง
FRONT_STOP  = 0.20   # m ใกล้มาก -> หยุดทันที
FRONT_SLOW  = 0.60   # m เริ่มชะลอ (ปรับความเร็วตาม front distance)

# ===================== GLOBAL STATES =====================
latest_tof1_mm = None   # ToF1 (mm)
odom_x, odom_y = 0.0, 0.0

# ===================== CALLBACKS =====================
def tof_cb(sub_info):
    """ sub_info = [tof1, tof2, tof3, tof4] (mm); ใช้ tof1 ด้านหน้า """
    global latest_tof1_mm
    try:
        latest_tof1_mm = float(sub_info[0])
    except Exception:
        latest_tof1_mm = None

def pos_cb(position_info):
    """ position_info: [x, y, z] (SDK EP) """
    global odom_x, odom_y
    odom_x = position_info[0]
    odom_y = position_info[1]

# ===================== SCAN HELPERS =====================
def measure_at_angle(ep_gimbal, yaw_deg):
    """หมุน gimbal ไป yaw_deg แล้วอ่าน ToF1 หลายครั้ง -> m"""
    ep_gimbal.moveto(pitch=0, yaw=yaw_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(DWELL_SEC)

    samples = []
    for _ in range(SAMPLE_PER_POSE):
        if latest_tof1_mm and latest_tof1_mm > 0:
            samples.append(latest_tof1_mm)
        time.sleep(1.0 / max(1, FREQ_HZ))
    if not samples:
        return None
    return mean(samples) / 1000.0  # mm -> m

def measure_direction(ep_gimbal, angles):
    vals = []
    for ang in angles:
        d = measure_at_angle(ep_gimbal, ang)
        if d is not None:
            vals.append(d)
    return mean(vals) if vals else None

def scan_three(ep_gimbal):
    d_left  = measure_direction(ep_gimbal, ANGLES_LEFT)
    d_front = measure_direction(ep_gimbal, ANGLES_FRONT)
    d_right = measure_direction(ep_gimbal, ANGLES_RIGHT)
    # กลับ gimbal มาข้างหน้า
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    return d_left, d_front, d_right

# ===================== UTIL =====================
def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))

def dynamic_target_from_front_tof():
    """คำนวณ target ระยะเดินหน้า = clamp(ToF(front) - offset) หน่วย m"""
    if latest_tof1_mm and latest_tof1_mm > 0:
        tof_m = latest_tof1_mm / 1000.0
        target = clamp(tof_m - TOF_OFFSET_M, TARGET_MIN, TARGET_MAX)
        print(f"[TARGET] front_tof={tof_m:.2f} m -> target={target:.2f} m (offset {TOF_OFFSET_M:.2f})")
        return target
    # fallback
    target = 1.2
    print(f"[TARGET] no ToF, fallback={target:.2f} m")
    return target

# ===================== TURN & FACE =====================
def choose_and_face_best(ep_chassis, ep_gimbal):
    """
    1) สแกนซ้าย-หน้า-ขวา
    2) เลือกทิศที่ไกลสุด
    3) หมุน gimbal โชว์ทิศ แล้วหมุนตัวหุ่นไปทิศนั้น
    4) recenter gimbal
    คืนค่า: "front" เสมอหลังหมุนเสร็จ (เพราะหันตัวแล้ว) + ระยะที่ไกลสุด (สำหรับ log)
    """
    d_left, d_front, d_right = scan_three(ep_gimbal)
    print(f"[SCAN] L={None if d_left is None else round(d_left,2)}  "
          f"F={None if d_front is None else round(d_front,2)}  "
          f"R={None if d_right is None else round(d_right,2)}")

    candidates = []
    if d_left  is not None:  candidates.append(("left",  d_left))
    if d_front is not None:  candidates.append(("front", d_front))
    if d_right is not None:  candidates.append(("right", d_right))

    if not candidates:
        print("[WARN] ไม่มีข้อมูล ToF จากทั้งสามทิศ")
        return "front", None

    best_dir, best_dist = max(candidates, key=lambda x: x[1])
    print(f"[DECIDE] ทิศที่ไกลสุด: {best_dir} ~ {best_dist:.2f} m")

    turn_deg = {"left": +90, "front": 0, "right": -90}[best_dir]

    # หัน gimbal ไปโชว์ทิศก่อน
    ep_gimbal.moveto(pitch=0, yaw=turn_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # หมุนตัวหุ่นไปทิศนั้น
    if turn_deg != 0:
        ep_chassis.move(x=0, y=0, z=turn_deg, z_speed=Z_SPEED).wait_for_completed()

    # กลับ gimbal มาข้างหน้าของหุ่น (หลังหันแล้ว = มองไปทิศที่จะวิ่ง)
    ep_gimbal.recenter().wait_for_completed()

    return "front", best_dist

# ===================== PID FORWARD =====================
def move_forward_pid(ep_chassis, target_m, slow_by_front=True):
    """
    เดินหน้าแบบ PID ด้วย feedback จาก "ระยะทางที่เดินจริง" = hypot(dx, dy)
    - target_m: ระยะเป้าหมาย (เมตร)
    - slow_by_front: ชะลอ/หยุดตาม ToF ด้านหน้าแบบเรียลไทม์
    """
    global odom_x, odom_y

    x0, y0 = odom_x, odom_y
    integral = 0.0
    prev_err = target_m  # เริ่มต้นถือว่าเดิน 0 -> error = target
    t_prev = time.time()

    print(f"[PID] target={target_m:.2f} m (เดินหน้า)")
    while True:
        # ระยะที่เดินมาแล้ว
        dx = odom_x - x0
        dy = odom_y - y0
        traveled = hypot(dx, dy)
        error = target_m - traveled

        # หยุดเมื่อถึงเป้า
        if abs(error) <= 0.01:
            ep_chassis.drive_speed(x=0, y=0, z=0)
            print("[PID] Reached target.")
            break

        # PID terms
        now = time.time()
        dt = max(1e-3, now - t_prev)
        t_prev = now

        integral += error * dt
        derivative = (error - prev_err) / dt
        prev_err = error

        cmd = Kp * error + Ki * integral + Kd * derivative

        # ชะลอตาม front ToF แบบเรียลไทม์ (ไม่ให้พุ่งเข้ากำแพง)
        if slow_by_front and latest_tof1_mm and latest_tof1_mm > 0:
            front_m = latest_tof1_mm / 1000.0
            if front_m <= FRONT_STOP:
                cmd = 0.0  # หยุดฉุกเฉิน
            elif front_m < FRONT_SLOW:
                # สเกลความเร็วลงตามสัดส่วน
                ratio = (front_m - FRONT_STOP) / (FRONT_SLOW - FRONT_STOP)
                cmd *= clamp(ratio, 0.0, 1.0)

        # ลิมิตความเร็ว
        cmd = clamp(cmd, V_MIN, V_MAX)

        ep_chassis.drive_speed(x=cmd, y=0.0, z=0.0)
        print(f"[PID] err={error:.3f} m, v={cmd:.3f} m/s, traveled={traveled:.3f} m")
        time.sleep(DT)

# ===================== MAIN =====================
def main():
    ep = robot.Robot()
    ep.initialize(conn_type=CONN_TYPE)

    ep_chassis = ep.chassis
    ep_gimbal  = ep.gimbal
    ep_sensor  = ep.sensor

    # ตั้งค่าเริ่ม
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # สมัคร callback
    ep_sensor.sub_distance(freq=FREQ_HZ, callback=tof_cb)
    ep_chassis.sub_position(freq=10, callback=pos_cb)

    print("เริ่ม: สแกนซ้าย-หน้า-ขวา → หันไปทิศที่ไกลสุด → เดินหน้าแบบ PID (target = ToFfront - 0.30 m)")
    try:
        while True:
            # 1) เลือกทิศที่ไกลสุดและหันตัวไปทางนั้น
            _, _best = choose_and_face_best(ep_chassis, ep_gimbal)

            # 2) อ่าน ToF ด้านหน้าหลังหันเสร็จ แล้วตั้งเป้าหมายเดินหน้า
            time.sleep(0.2)  # เว้นให้ ToF อัปเดตเฟรมใหม่
            target = dynamic_target_from_front_tof()

            # 3) เดินหน้าแบบ PID ตาม target
            move_forward_pid(ep_chassis, target_m=target, slow_by_front=True)

            # 4) เว้นช่วงสั้น ๆ แล้ววนสแกนรอบใหม่
            time.sleep(0.3)

    except KeyboardInterrupt:
        print("หยุดตามคำสั่งผู้ใช้")

    finally:
        try:
            ep_chassis.drive_speed(x=0, y=0, z=0)
        except Exception:
            pass
        try:
            ep_sensor.unsub_distance()
            ep_chassis.unsub_position()
        except Exception:
            pass
        try:
            ep_gimbal.recenter().wait_for_completed()
        except Exception:
            pass
        ep.close()

if __name__ == "__main__":
    main()


เริ่ม: สแกนซ้าย-หน้า-ขวา → หันไปทิศที่ไกลสุด → เดินหน้าแบบ PID (target = ToFfront - 0.30 m)
[SCAN] L=0.19  F=1.46  R=0.18
[DECIDE] ทิศที่ไกลสุด: front ~ 1.46 m
[TARGET] front_tof=1.45 m -> target=1.15 m (offset 0.30)
[PID] target=1.15 m (เดินหน้า)
[PID] err=1.152 m, v=0.800 m/s, traveled=0.000 m
[PID] err=1.152 m, v=0.800 m/s, traveled=0.000 m
[PID] err=1.130 m, v=0.800 m/s, traveled=0.022 m
[PID] err=1.060 m, v=0.800 m/s, traveled=0.092 m
[PID] err=0.980 m, v=0.800 m/s, traveled=0.172 m
[PID] err=0.887 m, v=0.800 m/s, traveled=0.265 m
[PID] err=0.799 m, v=0.800 m/s, traveled=0.353 m
[PID] err=0.723 m, v=0.800 m/s, traveled=0.429 m
[PID] err=0.650 m, v=0.800 m/s, traveled=0.502 m
[PID] err=0.571 m, v=0.800 m/s, traveled=0.581 m
[PID] err=0.488 m, v=0.800 m/s, traveled=0.664 m
[PID] err=0.405 m, v=0.800 m/s, traveled=0.747 m
[PID] err=0.327 m, v=0.521 m/s, traveled=0.825 m
[PID] err=0.249 m, v=0.262 m/s, traveled=0.903 m
[PID] err=0.170 m, v=0.101 m/s, traveled=0.982 m
[PID] err=0.103 m

In [10]:
import time
from statistics import mean
from robomaster import robot

# =============== CONFIG ===============
CONN_TYPE = "ap"

# Scan / gimbal
FREQ_HZ = 10
SAMPLE_PER_POSE = 2       # อ่านซ้ำต่อมุมเพื่อลด noise (ยิ่งมากยิ่งนิ่ง แต่สแกนนาน)
DWELL_SEC = 0.06          # รอให้ gimbal นิ่งก่อนอ่าน
GIMBAL_SPEED = 240        # deg/s
Z_SPEED = 100             # deg/s (หมุนตัวหุ่น)
RECHECK_SEC = 1.0         # สแกนซ้ำระหว่างวิ่งทุก ๆ กี่วินาที

# มุมสแกน
ANGLES_LEFT  = [-90]
ANGLES_FRONT = [0]
ANGLES_RIGHT = [90]

# Forward PID (ควบคุม "ความเร็ว" เดินหน้า)
Kp = 2
Ki = 0.1
Kd = 0.1
DT = 0.10                 # คาบควบคุม
V_MAX = 0.8               # m/s
V_MIN = 0.0               # m/s (ไม่ถอยหลังในโหมดนี้)

# นโยบายเว้นระยะด้านหน้า
FRONT_BUFFER = 0.01       # อยากให้เหลือบัฟเฟอร์
E_STOP = 0.20             # m ใกล้มาก -> หยุดฉุกเฉิน
SLOW_START = 0.30         # m ชะลอเมื่อเข้าโซนนี้

# เกณฑ์ “ต่างกันชัดเจน” ระหว่างทิศที่ชนะกับทิศปัจจุบัน
DIR_SWITCH_MARGIN = 0.20  # m เกินกว่านี้ค่อยเปลี่ยนทิศ

# =============== GLOBAL STATE ===============
latest_tof1_mm = None
heading_deg = 0           # ตำแหน่งหัว (0=หน้า, +90=ซ้าย, -90=ขวา) เราจะหมุนทีละ 90°

# =============== CALLBACKS ===============
def tof_cb(sub_info):
    global latest_tof1_mm
    try:
        latest_tof1_mm = float(sub_info[0])  # tof1 (mm)
    except:
        latest_tof1_mm = None

# =============== HELPERS ===============
def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))

def fmt(d):
    return "None" if d is None else f"{d:.2f}"

# ---- สแกน ----
def measure_at_angle(ep_gimbal, yaw_deg):
    ep_gimbal.moveto(pitch=0, yaw=yaw_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(DWELL_SEC)
    samples = []
    for _ in range(SAMPLE_PER_POSE):
        if latest_tof1_mm and latest_tof1_mm > 0:
            samples.append(latest_tof1_mm)
        time.sleep(1.0 / max(1, FREQ_HZ))
    if not samples:
        return None
    return mean(samples) / 1000.0  # m

def measure_dir(ep_gimbal, angles):
    vals = []
    for ang in angles:
        d = measure_at_angle(ep_gimbal, ang)
        if d is not None:
            vals.append(d)
    return mean(vals) if vals else None

def quick_scan(ep_gimbal):
    d_left  = measure_dir(ep_gimbal, ANGLES_LEFT)
    d_front = measure_dir(ep_gimbal, ANGLES_FRONT)
    d_right = measure_dir(ep_gimbal, ANGLES_RIGHT)
    # กลับ gimbal มาหน้า
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    print(f"[SCAN] L={fmt(d_left)} F={fmt(d_front)} R={fmt(d_right)}")
    return d_left, d_front, d_right

def choose_best_direction(d_left, d_front, d_right):
    cands = []
    if d_left  is not None:  cands.append(("left", d_left))
    if d_front is not None:  cands.append(("front", d_front))
    if d_right is not None:  cands.append(("right", d_right))
    if not cands:
        return None, None
    return max(cands, key=lambda x: x[1])  # (dir, dist)

def dir_to_abs_angle(d):
    # กำหนดมุมอ้างอิง: 0=front, +90=left, -90=right
    return {"front": 0, "left": +90, "right": -90}[d]

def turn_to_direction(ep_chassis, ep_gimbal, target_dir):
    global heading_deg
    target_ang = dir_to_abs_angle(target_dir)
    delta = target_ang - heading_deg
    if delta != 0:
        # โชว์ gimbal ไปทิศที่จะหัน (optional)
        ep_gimbal.moveto(pitch=0, yaw=delta,
                         pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
        # หมุนตัว
        ep_chassis.move(x=0, y=0, z=delta, z_speed=Z_SPEED).wait_for_completed()
        heading_deg = target_ang
        ep_gimbal.recenter().wait_for_completed()
    print(f"[TURN] -> {target_dir} ({target_ang}°)")

# ---- PID forward (คุมความเร็ว) ----
class PIDV:
    def __init__(self, Kp, Ki, Kd, vmin=0.0, vmax=0.8):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.vmin, self.vmax = vmin, vmax
        self.i = 0.0
        self.prev_e = None

    def reset(self):
        self.i = 0.0
        self.prev_e = None

    def step(self, free_space, dt):
        """
        free_space = ระยะว่างด้านหน้าเหนือ buffer (>=0)
        คอนโทรลเอาต์พุต = ความเร็วเดินหน้า (m/s)
        """
        e = free_space  # อยากให้มาก -> เร็ว, น้อย -> ช้า/หยุด
        self.i += e * dt
        d = 0.0 if self.prev_e is None else (e - self.prev_e) / max(1e-6, dt)
        self.prev_e = e
        v = self.Kp*e + self.Ki*self.i + self.Kd*d
        return clamp(v, self.vmin, self.vmax)

pid_v = PIDV(Kp, Ki, Kd, vmin=V_MIN, vmax=V_MAX)

def compute_forward_speed():
    """
    คำนวณความเร็วเดินหน้าจาก ToF หน้า:
    - ถ้า ToF <= E_STOP -> 0 (หยุดฉุกเฉิน)
    - มิฉะนั้นใช้ PID บน free_space = max(0, front - FRONT_BUFFER)
    - ลดความเร็วเพิ่มเติมเมื่อเข้าช่วง SLOW_START
    """
    if not latest_tof1_mm or latest_tof1_mm <= 0:
        # ไม่รู้ระยะหน้า -> เดินช้า ๆ เพื่อความปลอดภัย
        return 0.15

    front_m = latest_tof1_mm / 1000.0
    if front_m <= E_STOP:
        return 0.0

    free_space = max(0.0, front_m - FRONT_BUFFER )
    v = pid_v.step(free_space, DT)

    if front_m < SLOW_START:
        ratio = (front_m - E_STOP) / max(1e-6, (SLOW_START - E_STOP))
        v = v * clamp(ratio, 0.0, 1.0)

    return clamp(v, V_MIN, V_MAX)

# =============== MAIN BEHAVIOR LOOP ===============
def navigate_forever(ep_chassis, ep_gimbal):
    global heading_deg
    heading_deg = 0
    pid_v.reset()

    last_scan_t = 0
    current_dir = None
    print("เริ่มโหมด: สแกน → เลือกทิศไกลสุด → หมุน → วิ่งด้วย PID → รีเช็ควนไป (Ctrl+C เพื่อหยุด)")

    while True:
        now = time.time()

        # ต้องรีเช็คไหม?
        do_rescan = (current_dir is None) or (now - last_scan_t >= RECHECK_SEC)

        if do_rescan:
            # แช่ความเร็วไว้ชั่วครู่ (กัน jitter ตอน gimbal กวาด)
            ep_chassis.drive_speed(x=0.0, y=0.0, z=0.0)

            d_left, d_front, d_right = quick_scan(ep_gimbal)
            best_dir, best_dist = choose_best_direction(d_left, d_front, d_right)

            if best_dir is None:
                print("[WARN] ไม่มีผลสแกน ใช้ทิศเดิม/หยุดรอเฟรมใหม่")
                time.sleep(0.2)
                last_scan_t = time.time()
                continue

            # ถ้าเพิ่งเริ่ม หรือทิศใหม่ชนะทิศเดิมชัดเจน -> หมุนไปทิศใหม่
            if (current_dir is None) or (best_dir != current_dir):
                # หากมีทิศเดิม และทิศใหม่ไม่ได้ชนะมากพอ ให้คงทิศเดิม (ลดการเปลี่ยนทิศถี่)
                if current_dir is not None:
                    prev_dist = {"left": d_left, "front": d_front, "right": d_right}[current_dir]
                    if (prev_dist is not None) and (best_dist - prev_dist < DIR_SWITCH_MARGIN):
                        print(f"[HOLD] คงทิศ {current_dir} (Δ={best_dist - prev_dist:.2f} < {DIR_SWITCH_MARGIN})")
                    else:
                        turn_to_direction(ep_chassis, ep_gimbal, best_dir)
                        current_dir = best_dir
                else:
                    turn_to_direction(ep_chassis, ep_gimbal, best_dir)
                    current_dir = best_dir

            last_scan_t = time.time()

        # คุมความเร็วเดินหน้าแบบต่อเนื่อง
        v = compute_forward_speed()

        # ถ้าใกล้มาก -> หยุดแล้วบังคับรีเช็คทันทีรอบถัดไป
        if v <= 0.0:
            ep_chassis.drive_speed(x=0.0, y=0.0, z=0.0)
            # รีเช็คเร็วขึ้นเผื่อจะเปลี่ยนทิศได้
            last_scan_t = 0
            time.sleep(DT)
            continue

        ep_chassis.drive_speed(x=v, y=0.0, z=0.0)
        # debug
        fm = latest_tof1_mm/1000.0 if latest_tof1_mm else None
        print(f"[RUN] dir={current_dir}  v={v:.2f} m/s  front={fmt(fm)} m")
        time.sleep(DT)

# =============== ENTRY POINT ===============
def main():
    ep = robot.Robot()
    ep.initialize(conn_type=CONN_TYPE)

    ep_chassis = ep.chassis
    ep_gimbal  = ep.gimbal
    ep_sensor  = ep.sensor

    # ตั้ง gimbal หันหน้า
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # สมัครข้อมูล ToF
    ep_sensor.sub_distance(freq=FREQ_HZ, callback=tof_cb)

    try:
        navigate_forever(ep_chassis, ep_gimbal)
    except KeyboardInterrupt:
        print("หยุดตามคำสั่งผู้ใช้")
    finally:
        try:
            ep_chassis.drive_speed(x=0, y=0, z=0)
        except Exception:
            pass
        try:
            ep_sensor.unsub_distance()
        except Exception:
            pass
        try:
            ep_gimbal.recenter().wait_for_completed()
        except Exception:
            pass
        ep.close()

if __name__ == "__main__":
    main()


เริ่มโหมด: สแกน → เลือกทิศไกลสุด → หมุน → วิ่งด้วย PID → รีเช็ควนไป (Ctrl+C เพื่อหยุด)
[SCAN] L=0.16 F=1.23 R=0.27
[TURN] -> front (0°)
[RUN] dir=front  v=0.80 m/s  front=0.53 m
[RUN] dir=front  v=0.80 m/s  front=1.63 m
[RUN] dir=front  v=0.80 m/s  front=1.52 m
[RUN] dir=front  v=0.80 m/s  front=1.49 m
[RUN] dir=front  v=0.80 m/s  front=1.40 m
[RUN] dir=front  v=0.80 m/s  front=1.40 m
[RUN] dir=front  v=0.80 m/s  front=1.26 m
[RUN] dir=front  v=0.80 m/s  front=1.26 m
[RUN] dir=front  v=0.80 m/s  front=1.04 m
[RUN] dir=front  v=0.80 m/s  front=0.91 m
[SCAN] L=0.07 F=0.56 R=1.12
หยุดตามคำสั่งผู้ใช้


In [2]:
import time
from statistics import mean
from robomaster import robot

# =============== CONFIG ===============
CONN_TYPE = "ap"

# Scan / gimbal
FREQ_HZ = 10
SAMPLE_PER_POSE = 2
DWELL_SEC = 0.06
GIMBAL_SPEED = 240
Z_SPEED = 100
RECHECK_SEC = 1.0

# มุมสแกน
ANGLES_LEFT  = [-90]
ANGLES_FRONT = [0]
ANGLES_RIGHT = [90]

# Forward PID
Kp = 2
Ki = 0.1
Kd = 0.1
DT = 0.10
V_MAX = 0.8
V_MIN = 0.0

# นโยบายเว้นระยะด้านหน้า
FRONT_BUFFER = 0.01
E_STOP = 0.20
SLOW_START = 0.30
EXTRA_ADVANCE_M = 0.20  # เดินลึกกว่าเดิมอีก 0.20 m

# เกณฑ์เปลี่ยนทิศ
DIR_SWITCH_MARGIN = 0.20

# =============== GLOBAL STATE ===============
latest_tof1_mm = None
heading_deg = 0

# =============== CALLBACKS ===============
def tof_cb(sub_info):
    global latest_tof1_mm
    try:
        latest_tof1_mm = float(sub_info[0])
    except:
        latest_tof1_mm = None

# =============== HELPERS ===============
def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))

def fmt(d):
    return "None" if d is None else f"{d:.2f}"

# ---- สแกน ----
def measure_at_angle(ep_gimbal, yaw_deg):
    ep_gimbal.moveto(pitch=0, yaw=yaw_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(DWELL_SEC)
    samples = []
    for _ in range(SAMPLE_PER_POSE):
        if latest_tof1_mm and latest_tof1_mm > 0:
            samples.append(latest_tof1_mm)
        time.sleep(1.0 / max(1, FREQ_HZ))
    if not samples:
        return None
    return mean(samples) / 1000.0  # m

def measure_dir(ep_gimbal, angles):
    vals = []
    for ang in angles:
        d = measure_at_angle(ep_gimbal, ang)
        if d is not None:
            vals.append(d)
    return mean(vals) if vals else None

def quick_scan(ep_gimbal):
    d_left  = measure_dir(ep_gimbal, ANGLES_LEFT)
    d_front = measure_dir(ep_gimbal, ANGLES_FRONT)
    d_right = measure_dir(ep_gimbal, ANGLES_RIGHT)
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    print(f"[SCAN] L={fmt(d_left)} F={fmt(d_front)} R={fmt(d_right)}")
    return d_left, d_front, d_right

def choose_best_direction(d_left, d_front, d_right):
    cands = []
    if d_left  is not None:  cands.append(("left", d_left))
    if d_front is not None:  cands.append(("front", d_front))
    if d_right is not None:  cands.append(("right", d_right))
    if not cands:
        return None, None
    return max(cands, key=lambda x: x[1])

def dir_to_abs_angle(d):
    return {"front": 0, "left": +90, "right": -90}[d]

def turn_to_direction(ep_chassis, ep_gimbal, target_dir):
    global heading_deg
    target_ang = dir_to_abs_angle(target_dir)
    delta = target_ang - heading_deg
    if delta != 0:
        ep_gimbal.moveto(pitch=0, yaw=delta,
                         pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
        ep_chassis.move(x=0, y=0, z=delta, z_speed=Z_SPEED).wait_for_completed()
        heading_deg = target_ang
        ep_gimbal.recenter().wait_for_completed()
    print(f"[TURN] -> {target_dir} ({target_ang}°)")

# ---- PID forward ----
class PIDV:
    def __init__(self, Kp, Ki, Kd, vmin=0.0, vmax=0.8):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.vmin, self.vmax = vmin, vmax
        self.i = 0.0
        self.prev_e = None

    def reset(self):
        self.i = 0.0
        self.prev_e = None

    def step(self, free_space, dt):
        e = free_space
        self.i += e * dt
        d = 0.0 if self.prev_e is None else (e - self.prev_e) / max(1e-6, dt)
        self.prev_e = e
        v = self.Kp*e + self.Ki*self.i + self.Kd*d
        return clamp(v, self.vmin, self.vmax)

pid_v = PIDV(Kp, Ki, Kd, vmin=V_MIN, vmax=V_MAX)

def compute_forward_speed():
    if not latest_tof1_mm or latest_tof1_mm <= 0:
        return 0.15

    front_m = latest_tof1_mm / 1000.0
    if front_m <= E_STOP:
        return 0.0

    # ปรับบัฟเฟอร์ให้เดินลึกกว่าเดิม 0.2 m
    effective_buffer = max(E_STOP, FRONT_BUFFER - EXTRA_ADVANCE_M)
    free_space = max(0.0, front_m - effective_buffer)

    v = pid_v.step(free_space, DT)

    if front_m < SLOW_START:
        ratio = (front_m - E_STOP) / max(1e-6, (SLOW_START - E_STOP))
        v *= clamp(ratio, 0.0, 1.0)

    return clamp(v, V_MIN, V_MAX)

# =============== MAIN LOOP ===============
def navigate_forever(ep_chassis, ep_gimbal):
    global heading_deg
    heading_deg = 0
    pid_v.reset()
    last_scan_t = 0
    current_dir = None

    print("เริ่มโหมด: สแกน → เลือกทิศ → หมุน → เดิน PID → รีเช็ควนไป")

    while True:
        now = time.time()
        do_rescan = (current_dir is None) or (now - last_scan_t >= RECHECK_SEC)

        if do_rescan:
            ep_chassis.drive_speed(x=0.0, y=0.0, z=0.0)
            d_left, d_front, d_right = quick_scan(ep_gimbal)
            best_dir, best_dist = choose_best_direction(d_left, d_front, d_right)

            if best_dir is None:
                print("[WARN] ไม่มีผลสแกน")
                time.sleep(0.2)
                last_scan_t = time.time()
                continue

            if (current_dir is None) or (best_dir != current_dir):
                if current_dir is not None:
                    prev_dist = {"left": d_left, "front": d_front, "right": d_right}[current_dir]
                    if (prev_dist is not None) and (best_dist - prev_dist < DIR_SWITCH_MARGIN):
                        print(f"[HOLD] คงทิศ {current_dir}")
                    else:
                        turn_to_direction(ep_chassis, ep_gimbal, best_dir)
                        current_dir = best_dir
                else:
                    turn_to_direction(ep_chassis, ep_gimbal, best_dir)
                    current_dir = best_dir

            last_scan_t = time.time()

        v = compute_forward_speed()

        if v <= 0.0:
            ep_chassis.drive_speed(x=0.0, y=0.0, z=0.0)
            last_scan_t = 0
            time.sleep(DT)
            continue

        ep_chassis.drive_speed(x=v, y=0.0, z=0.0)
        fm = latest_tof1_mm/1000.0 if latest_tof1_mm else None
        print(f"[RUN] dir={current_dir}  v={v:.2f} m/s  front={fmt(fm)} m")
        time.sleep(DT)

# =============== ENTRY POINT ===============
def main():
    ep = robot.Robot()
    ep.initialize(conn_type=CONN_TYPE)

    ep_chassis = ep.chassis
    ep_gimbal  = ep.gimbal
    ep_sensor  = ep.sensor

    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    ep_sensor.sub_distance(freq=FREQ_HZ, callback=tof_cb)

    try:
        navigate_forever(ep_chassis, ep_gimbal)
    except KeyboardInterrupt:
        print("หยุดตามคำสั่งผู้ใช้")
    finally:
        try: ep_chassis.drive_speed(x=0, y=0, z=0)
        except: pass
        try: ep_sensor.unsub_distance()
        except: pass
        try: ep_gimbal.recenter().wait_for_completed()
        except: pass
        ep.close()

if __name__ == "__main__":
    main()


เริ่มโหมด: สแกน → เลือกทิศ → หมุน → เดิน PID → รีเช็ควนไป
[SCAN] L=0.27 F=1.12 R=0.16
[TURN] -> front (0°)
[RUN] dir=front  v=0.80 m/s  front=1.43 m
[RUN] dir=front  v=0.80 m/s  front=1.40 m
[RUN] dir=front  v=0.80 m/s  front=1.40 m
[RUN] dir=front  v=0.80 m/s  front=1.39 m
[RUN] dir=front  v=0.80 m/s  front=1.33 m
[RUN] dir=front  v=0.80 m/s  front=1.25 m
[RUN] dir=front  v=0.80 m/s  front=1.05 m
[RUN] dir=front  v=0.80 m/s  front=1.05 m
[RUN] dir=front  v=0.80 m/s  front=0.89 m
[RUN] dir=front  v=0.80 m/s  front=0.89 m
[SCAN] L=0.14 F=0.33 R=1.53
[TURN] -> right (-90°)
[RUN] dir=right  v=0.80 m/s  front=1.54 m
[RUN] dir=right  v=0.80 m/s  front=1.54 m
[RUN] dir=right  v=0.80 m/s  front=1.53 m
[RUN] dir=right  v=0.80 m/s  front=1.52 m
[RUN] dir=right  v=0.80 m/s  front=1.41 m
[RUN] dir=right  v=0.80 m/s  front=1.31 m
[RUN] dir=right  v=0.80 m/s  front=1.21 m
[RUN] dir=right  v=0.80 m/s  front=1.21 m
[RUN] dir=right  v=0.80 m/s  front=1.05 m
[RUN] dir=right  v=0.80 m/s  front=0.97 m
[S

In [ ]:
import time
from statistics import mean
from math import hypot
from robomaster import robot

# =============== CONFIG ===============
CONN_TYPE = "ap"

# Scan / gimbal
FREQ_HZ = 10
SAMPLE_PER_POSE = 2
DWELL_SEC = 0.06
GIMBAL_SPEED = 240
Z_SPEED = 100
RECHECK_SEC = 1.0           # เว้นก่อนสแกนรอบใหม่หลังจบสเต็ป

# มุมสแกน
ANGLES_LEFT  = [-90]
ANGLES_FRONT = [0]
ANGLES_RIGHT = [90]

# Forward PID (สำหรับ "สเต็ป" ทีละระยะ)
Kp = 2.0
Ki = 0.1
Kd = 0.1
DT = 0.10                   # คาบควบคุม
V_MAX = 0.8
V_MIN = 0.0

# ระยะสเต็ปต่อครั้ง
STEP_M = 0.60               # <<< เดินทีละ 0.6 m

# นโยบายเว้นระยะด้านหน้า (สำหรับความปลอดภัยระหว่างสเต็ป)
FRONT_BUFFER = 0.10         # ระยะที่อยากเหลือไว้หน้าเซ็นเซอร์
E_STOP = 0.20               # ใกล้มาก -> หยุดฉุกเฉิน
SLOW_START = 0.30           # เริ่มชะลอเมื่อเข้าโซนนี้
EXTRA_ADVANCE_M = 0.20      # อยากให้ลึกกว่า buffer เดิมอีก 0.2 m (จะถูกคุมไม่ให้ต่ำกว่า E_STOP)

# เกณฑ์เปลี่ยนทิศ
DIR_SWITCH_MARGIN = 0.20

# =============== GLOBAL STATE ===============
latest_tof1_mm = None
heading_deg = 0
odom_x, odom_y = 0.0, 0.0   # odometry จากตัวหุ่น

# =============== CALLBACKS ===============
def tof_cb(sub_info):
    global latest_tof1_mm
    try:
        latest_tof1_mm = float(sub_info[0])  # tof1 (mm)
    except:
        latest_tof1_mm = None

def pos_cb(position_info):
    global odom_x, odom_y
    odom_x = position_info[0]
    odom_y = position_info[1]

# =============== HELPERS ===============
def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))

def fmt(d):
    return "None" if d is None else f"{d:.2f}"

# ---- สแกน ----
def measure_at_angle(ep_gimbal, yaw_deg):
    ep_gimbal.moveto(pitch=0, yaw=yaw_deg,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(DWELL_SEC)
    samples = []
    for _ in range(SAMPLE_PER_POSE):
        if latest_tof1_mm and latest_tof1_mm > 0:
            samples.append(latest_tof1_mm)
        time.sleep(1.0 / max(1, FREQ_HZ))
    if not samples:
        return None
    return mean(samples) / 1000.0  # m

def measure_dir(ep_gimbal, angles):
    vals = []
    for ang in angles:
        d = measure_at_angle(ep_gimbal, ang)
        if d is not None:
            vals.append(d)
    return mean(vals) if vals else None

def quick_scan(ep_gimbal):
    d_left  = measure_dir(ep_gimbal, ANGLES_LEFT)
    d_front = measure_dir(ep_gimbal, ANGLES_FRONT)
    d_right = measure_dir(ep_gimbal, ANGLES_RIGHT)
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    print(f"[SCAN] L={fmt(d_left)} F={fmt(d_front)} R={fmt(d_right)}")
    return d_left, d_front, d_right

def choose_best_direction(d_left, d_front, d_right):
    cands = []
    if d_left  is not None:  cands.append(("left", d_left))
    if d_front is not None:  cands.append(("front", d_front))
    if d_right is not None:  cands.append(("right", d_right))
    if not cands:
        return None, None
    return max(cands, key=lambda x: x[1])

def dir_to_abs_angle(d):
    return {"front": 0, "left": +90, "right": -90}[d]

def turn_to_direction(ep_chassis, ep_gimbal, target_dir):
    global heading_deg
    target_ang = dir_to_abs_angle(target_dir)
    delta = target_ang - heading_deg
    if delta != 0:
        ep_gimbal.moveto(pitch=0, yaw=delta,
                         pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()
        ep_chassis.move(x=0, y=0, z=delta, z_speed=Z_SPEED).wait_for_completed()
        heading_deg = target_ang
        ep_gimbal.recenter().wait_for_completed()
    print(f"[TURN] -> {target_dir} ({target_ang}°)")

# ---- PID step (เดินไปให้ครบ STEP_M ด้วย odometry + เช็ค ToF ตลอด) ----
def move_forward_step_pid(ep_chassis, step_m):
    print(f"[STEP] target={step_m:.2f} m")
    # PID ภายในสเต็ป (positional control)
    integral = 0.0
    prev_e = step_m
    t_prev = time.time()
    x0, y0 = odom_x, odom_y

    while True:
        traveled = hypot(odom_x - x0, odom_y - y0)
        error = step_m - traveled

        if error <= 0.01:
            ep_chassis.drive_speed(x=0.0, y=0.0, z=0.0)
            print(f"[STEP] done, traveled={traveled:.2f} m")
            break

        now = time.time()
        dt = max(1e-3, now - t_prev)
        t_prev = now

        integral += error * dt
        deriv = (error - prev_e) / dt
        prev_e = error

        v = Kp*error + Ki*integral + Kd*deriv
        v = clamp(v, V_MIN, V_MAX)

        # Safety จาก ToF หน้า
        fm = latest_tof1_mm/1000.0 if latest_tof1_mm else None
        if fm is not None:
            if fm <= E_STOP:
                ep_chassis.drive_speed(x=0.0, y=0.0, z=0.0)
                print("[E-STOP] front too close, abort step")
                break
            if fm < SLOW_START:
                ratio = (fm - E_STOP) / max(1e-6, (SLOW_START - E_STOP))
                v *= clamp(ratio, 0.0, 1.0)

        ep_chassis.drive_speed(x=v, y=0.0, z=0.0)
        print(f"[RUN STEP] err={error:.2f} v={v:.2f} traveled={traveled:.2f}")
        time.sleep(DT)

# =============== MAIN LOOP (step-by-step) ===============
def navigate_by_steps(ep_chassis, ep_gimbal):
    global heading_deg
    heading_deg = 0
    print("โหมดสเต็ป: สแกน → เลือกทิศ → หมุน → เดิน 0.6 m → วนใหม่")

    while True:
        # 1) สแกนแล้วเลือกทิศไกลสุด
        d_left, d_front, d_right = quick_scan(ep_gimbal)
        best_dir, best_dist = choose_best_direction(d_left, d_front, d_right)
        if best_dir is None:
            print("[WARN] ไม่มีผลสแกน รอแล้วลองใหม่")
            time.sleep(0.2)
            continue

        # 2) หมุนไปทิศที่เลือก
        turn_to_direction(ep_chassis, ep_gimbal, best_dir)

        # 3) กำหนดระยะสเต็ปจริงโดยเผื่อ buffer (กันพุ่งติดกำแพง)
        fm = latest_tof1_mm/1000.0 if latest_tof1_mm else None
        effective_buffer = max(E_STOP, FRONT_BUFFER - EXTRA_ADVANCE_M)
        if fm is None:
            step_len = STEP_M
        else:
            max_allowed = max(0.0, fm - effective_buffer)
            step_len = clamp(STEP_M, 0.0, max_allowed)

        if step_len <= 0.0:
            print("[HOLD] front ใกล้เกิน เดินไม่ได้ในรอบนี้")
            time.sleep(0.2)
            continue

        # 4) เดินสเต็ปด้วย PID
        move_forward_step_pid(ep_chassis, step_len)

        # 5) เว้นก่อนเริ่มรอบใหม่
        time.sleep(RECHECK_SEC)

# =============== ENTRY POINT ===============
def main():
    ep = robot.Robot()
    ep.initialize(conn_type=CONN_TYPE)

    ep_chassis = ep.chassis
    ep_gimbal  = ep.gimbal
    ep_sensor  = ep.sensor

    # ตั้งกิมบอลหันหน้า
    ep_gimbal.moveto(pitch=0, yaw=0,
                     pitch_speed=GIMBAL_SPEED, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # สมัคร ToF + Odometry
    ep_sensor.sub_distance(freq=FREQ_HZ, callback=tof_cb)
    ep_chassis.sub_position(freq=10, callback=pos_cb)

    try:
        navigate_by_steps(ep_chassis, ep_gimbal)
    except KeyboardInterrupt:
        print("หยุดตามคำสั่งผู้ใช้")
    finally:
        try: ep_chassis.drive_speed(x=0, y=0, z=0)
        except: pass
        try:
            ep_sensor.unsub_distance()
            ep_chassis.unsub_position()
        except: pass
        try: ep_gimbal.recenter().wait_for_completed()
        except: pass
        ep.close()

if __name__ == "__main__":
    main()
